# 06B — SFLLD Train/Validation Row Filter + FE-Master Column Freeze

**Input**

`data/output/final_modeling_dataset/SFLLD_modeling_dataset_missing_values_treated.parquet`

**Output**

`data/output/final_modeling_dataset/SFLLD_feature_engineering_master_train_validation.parquet`

This notebook implements the frozen row/column policy:

- keep **Train + Validation** rows only;
- exclude OOT from this downstream artifact;
- retain both **Regular PIT** and **Strict PIT** final macro frameworks;
- retain State/MSA inputs required for later macro-regime feature engineering;
- retain control / QA / provenance fields explicitly approved;
- remove constant, near-unavailable, duplicated, superseded, fallback-intermediate, and 09F/09G construction fields;
- classify **every input column exactly once** and fail loudly if the actual schema differs from the frozen manifest;
- do **not** perform new feature engineering, imputation, scaling, clustering, or model fitting.

The source parquet is never overwritten.


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)
pd.set_option("display.max_rows", 250)

# ------------------------------------------------------------------
# Robust project-root discovery
# Works when notebook is run from project root OR /notebooks.
# ------------------------------------------------------------------
INPUT_NAME = "SFLLD_modeling_dataset_missing_values_treated.parquet"

def find_project_root():
    start = Path.cwd().resolve()
    candidates = [start, *start.parents]
    for root in candidates:
        p = root / "data" / "output" / "final_modeling_dataset" / INPUT_NAME
        if p.exists():
            return root
    searched = [
        str(r / "data" / "output" / "final_modeling_dataset" / INPUT_NAME)
        for r in candidates
    ]
    raise FileNotFoundError(
        "Could not locate project root from the current working directory.\n"
        "Expected to find:\n  " + "\n  ".join(searched)
    )

ROOT = find_project_root()

INPUT = (
    ROOT / "data" / "output" / "final_modeling_dataset"
    / "SFLLD_modeling_dataset_missing_values_treated.parquet"
)
OUTPUT = (
    ROOT / "data" / "output" / "final_modeling_dataset"
    / "SFLLD_feature_engineering_master_train_validation.parquet"
)
QA_DIR = (
    ROOT / "data" / "output" / "quality_assurance"
    / "06B_train_validation_FE_master_freeze"
)
QA_DIR.mkdir(parents=True, exist_ok=True)

print("Project root :", ROOT)
print("Input        :", INPUT)
print("Output       :", OUTPUT)
print("QA directory :", QA_DIR)
assert INPUT.exists(), f"Input not found: {INPUT}"

Project root : C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6
Input        : C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6\data\output\final_modeling_dataset\SFLLD_modeling_dataset_missing_values_treated.parquet
Output       : C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6\data\output\final_modeling_dataset\SFLLD_feature_engineering_master_train_validation.parquet
QA directory : C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6\data\output\quality_assurance\06B_train_validation_FE_master_freeze


## 1. Load source + immutable-source audit

In [2]:
df = pd.read_parquet(INPUT)

print(f"Source shape : {df.shape[0]:,} rows × {df.shape[1]:,} columns")
print("\nSplit counts:")
display(
    df["Split"]
    .astype("string")
    .value_counts(dropna=False)
    .rename_axis("Split")
    .reset_index(name="N")
)

# Core invariants before any reduction
CORE_REQUIRED = ["Loan_ID", "Split", "Target_24M"]
missing_core = [c for c in CORE_REQUIRED if c not in df.columns]
assert not missing_core, f"Missing required core columns: {missing_core}"

assert df["Loan_ID"].notna().all(), "Loan_ID contains missing values."
assert df["Loan_ID"].is_unique, "Loan_ID is not unique in the source dataset."
assert df["Split"].notna().all(), "Split contains missing values."

target_numeric = pd.to_numeric(df["Target_24M"], errors="coerce")
assert target_numeric.notna().all(), "Target_24M contains missing/non-numeric values."
assert set(target_numeric.unique()).issubset({0, 1}), \
    f"Unexpected Target_24M values: {sorted(target_numeric.unique().tolist())}"

source_rows = len(df)
source_cols = list(df.columns)
source_ncols = len(source_cols)


Source shape : 713,000 rows × 110 columns

Split counts:


,Split,N
0,Other Period,404240
1,Train,160003
2,OOT Test,99719
3,Validation,49038


## 2. Frozen schema manifest

Statuses:

- `MODEL_KEEP`: retained model/benchmark candidate.
- `KEEP_FE`: retained for downstream feature engineering or provenance diagnostics.
- `CONTROL_QA`: retained for identity/split/audit, not a predictor by default.
- `TARGET`: target only.
- `DROP`: removed from the FE-master artifact.

The manifest is intentionally explicit. **No unclassified source column is silently kept or dropped.**


In [3]:
STATUS = {}

def register(cols, status, block, reason):
    """Register explicit frozen decisions; duplicate registration is an error."""
    for c in cols:
        if c in STATUS:
            raise AssertionError(
                f"Column classified more than once: {c!r}\n"
                f"Existing={STATUS[c]}\nNew={(status, block, reason)}"
            )
        STATUS[c] = {
            "status": status,
            "block": block,
            "reason": reason,
        }

# ================================================================
# BLOCK 1 — Control and time fields
# ================================================================
register(
    ["Loan_ID", "Split"],
    "CONTROL_QA", "Block 1",
    "Identity / split control; retained but not a predictor by default."
)
register(
    ["Target_24M"],
    "TARGET", "Block 1",
    "24-month modeling target."
)
register(
    ["Dataset_Vintage", "First_Payment_Date", "Maturity_Date"],
    "KEEP_FE", "Block 1",
    "Time/vintage source retained for downstream feature engineering."
)
register(
    ["Performance_Window_End"],
    "CONTROL_QA", "Block 1",
    "Target-window audit field; must not enter predictor matrix."
)
register(
    ["COVID_Window_Overlap"],
    "MODEL_KEEP", "Block 1",
    "Existing period/challenger feature."
)

# ================================================================
# BLOCK 2 — Original loan-level fields
# ================================================================
register(
    [
        "FICO_Score",
        "First_Time_Homebuyer",
        "MI_Percentage",
        "Number_of_Units",
        "Occupancy_Status",
        "Original_CLTV",
        "Original_DTI",
        "Original_UPB",
        "Original_LTV",
        "Original_Interest_Rate",
        "Channel",
        "Property_Type",
        "Loan_Purpose",
        "Original_Loan_Term",
        "Number_of_Borrowers",
        "Seller_Name",
        "Super_Conforming_Flag",
        "HARP_Indicator",
        "DTI_Missing",
    ],
    "MODEL_KEEP", "Block 2",
    "Retained loan-level modeling / challenger feature source."
)
register(
    ["MSA", "Property_State", "Postal_Code"],
    "KEEP_FE", "Block 2",
    "Geography keys required for State/MSA/ZIP-related feature engineering."
)
register(
    [
        "Prepayment_Penalty_Indicator",
        "Amortization_Type",
        "Interest_Only_Indicator",
    ],
    "DROP", "Block 2",
    "Frozen removal: constant / zero-variance field."
)
register(
    [
        "Special_Eligibility_Program",
        "Property_Valuation_Method",
    ],
    "DROP", "Block 2",
    "Frozen removal: Train coverage is nearly absent."
)

# ================================================================
# BLOCK 3 — Regular PIT housing geography
# ================================================================
register(
    ["PIT_FHFA_Quarter_Date"],
    "KEEP_FE", "Block 3",
    "Regular PIT housing time anchor."
)
register(
    ["Freddie_State", "Freddie_MSA", "Freddie_ZIP3"],
    "DROP", "Block 3",
    "Duplicative standardized geography representation."
)
register(
    [
        "FHFA_State_HPI", "FHFA_State_HPI_Growth_12M",
        "FHFA_MSA_HPI", "FHFA_MSA_HPI_Growth_12M",
    ],
    "KEEP_FE", "Block 3",
    "State/MSA macro-regime inputs."
)
register(
    ["FHFA_ZIP3_HPI", "FHFA_ZIP3_HPI_Growth_12M"],
    "DROP", "Block 3",
    "No independent ZIP3 macro-regime planned; final resolved housing signal retained."
)
register(
    ["Housing_HPI", "Housing_HPI_Growth_12M"],
    "MODEL_KEEP", "Block 3",
    "Regular PIT final resolved housing production variables."
)
register(
    ["Housing_Geography_Source", "Housing_Fallback_Level"],
    "KEEP_FE", "Block 3",
    "Regular housing provenance / fallback diagnostics."
)

# ================================================================
# BLOCK 4 — Regular PIT BLS
# ================================================================
register(
    ["PIT_BLS_Month_Date"],
    "KEEP_FE", "Block 4",
    "Regular BLS PIT time anchor."
)
register(
    [
        "BLS_Unemployment_Rate",
        "BLS_Employment",
        "BLS_Labor_Force",
        "BLS_Employment_Growth_12M",
        "BLS_Unemployment_Rate_Change_12M",
        "Labor_Geography_Source",
    ],
    "DROP", "Block 4",
    "Regular pre-fallback / pre-final fields superseded by final resolved variables."
)
register(
    [
        "BLS_Unemployment_Rate_State_Fallback",
        "BLS_Employment_Growth_12M_State_Fallback",
        "BLS_Unemployment_Rate_Change_12M_State_Fallback",
    ],
    "DROP", "Block 4",
    "Regular State-fallback candidate fields already incorporated into Final variables."
)
register(
    [
        "BLS_Unemployment_Rate_Final",
        "BLS_Employment_Growth_12M_Final",
        "BLS_Unemployment_Rate_Change_12M_Final",
    ],
    "MODEL_KEEP", "Block 4",
    "Regular PIT final resolved BLS production variables."
)
register(
    [
        "BLS_State_Fallback_Applied",
        "BLS_Fallback_Level",
        "BLS_Geography_Source_Final",
    ],
    "KEEP_FE", "Block 4",
    "Regular final BLS provenance / fallback diagnostics."
)
register(
    ["BLS_Remaining_Missing_Reason"],
    "DROP", "Block 4",
    "Audit/intermediate missing-reason field."
)

# ================================================================
# BLOCK 5 — Strict PIT housing geography
# ================================================================
register(
    ["Strict_PIT_FHFA_Quarter_Date"],
    "KEEP_FE", "Block 5",
    "Strict PIT housing time anchor."
)
register(
    ["LS_Freddie_State", "LS_Freddie_ZIP3", "LS_Freddie_MSA"],
    "DROP", "Block 5",
    "Duplicative Strict standardized geography representation."
)
register(
    [
        "LS_FHFA_State_HPI", "LS_FHFA_State_HPI_Growth_12M",
        "LS_FHFA_MSA_HPI", "LS_FHFA_MSA_HPI_Growth_12M",
    ],
    "KEEP_FE", "Block 5",
    "Strict State/MSA macro-regime inputs."
)
register(
    ["LS_FHFA_ZIP3_HPI", "LS_FHFA_ZIP3_HPI_Growth_12M"],
    "DROP", "Block 5",
    "No separate Strict ZIP3 macro-regime planned."
)
register(
    ["LS_Housing_HPI", "LS_Housing_HPI_Growth_12M"],
    "MODEL_KEEP", "Block 5",
    "Strict PIT final resolved housing production variables."
)
register(
    ["LS_Housing_Geography_Source", "LS_Housing_Fallback_Level"],
    "KEEP_FE", "Block 5",
    "Strict housing provenance / fallback diagnostics."
)

# ================================================================
# BLOCK 6 — Strict PIT BLS
# ================================================================
register(
    ["Strict_PIT_BLS_Month_Date"],
    "KEEP_FE", "Block 6",
    "Strict BLS PIT time anchor."
)
register(
    [
        "LS_BLS_Unemployment_Rate",
        "LS_BLS_Employment",
        "LS_BLS_Labor_Force",
        "LS_BLS_Employment_Growth_12M",
        "LS_BLS_Unemployment_Rate_Change_12M",
        "LS_Labor_Geography_Source",
    ],
    "DROP", "Block 6",
    "Strict pre-fallback / pre-final fields superseded by final resolved variables."
)
register(
    [
        "LS_BLS_Unemployment_Rate_State_Fallback",
        "LS_BLS_Employment_Growth_12M_State_Fallback",
        "LS_BLS_Unemployment_Rate_Change_12M_State_Fallback",
    ],
    "DROP", "Block 6",
    "Strict State-fallback candidate fields already incorporated into Final variables."
)
register(
    [
        "LS_BLS_Unemployment_Rate_Final",
        "LS_BLS_Employment_Growth_12M_Final",
        "LS_BLS_Unemployment_Rate_Change_12M_Final",
    ],
    "MODEL_KEEP", "Block 6",
    "Strict PIT final resolved BLS production variables."
)
register(
    [
        "LS_BLS_Geography_Source_Final",
        "LS_BLS_Fallback_Level",
        "LS_BLS_State_Fallback_Applied",
    ],
    "KEEP_FE", "Block 6",
    "Strict final BLS provenance / fallback diagnostics."
)

# ================================================================
# BLOCK 7 — 09F/09G construction fields
# ================================================================
register(
    [
        "_State_FIPS",
        "Strict_PIT_BLS_State_Month_Date",
        "BLS_State_Assumed_Available_Date",
        "_State_BLS_Unemployment_Rate",
        "_State_BLS_Civilian_Labor_Force",
        "_State_BLS_Employment",
        "_State_BLS_Unemployment",
        "_State_BLS_Labor_Force_Participation_Rate",
        "_State_BLS_Employment_Population_Ratio",
        "09G_BLS_Remaining_Missing_Reason",
    ],
    "DROP", "Block 7",
    "09F/09G merge/lookup construction artifact; no downstream use after Final fields."
)

# ================================================================
# BLOCK 8 — Residual-missingness flags
# ================================================================
register(
    [
        "LS_Housing_ResidualMissing",
        "LS_BLS_ResidualMissing",
        "Housing_Regular_ResidualMissing",
        "BLS_Regular_ResidualMissing",
    ],
    "KEEP_FE", "Block 8",
    "Residual-missingness diagnostic; retained for macro-cluster cross-checks."
)

print(f"Frozen manifest entries: {len(STATUS):,}")


Frozen manifest entries: 110


## 3. Schema reconciliation — fail on any mismatch

In [4]:
source_set = set(source_cols)
manifest_set = set(STATUS)

unclassified_source = sorted(source_set - manifest_set)
manifest_not_in_source = sorted(manifest_set - source_set)

schema_reconciliation = pd.DataFrame({
    "metric": [
        "source_column_count",
        "manifest_column_count",
        "unclassified_source_count",
        "manifest_not_in_source_count",
    ],
    "value": [
        len(source_set),
        len(manifest_set),
        len(unclassified_source),
        len(manifest_not_in_source),
    ],
})
display(schema_reconciliation)

if unclassified_source:
    print("\nUNCLASSIFIED SOURCE COLUMNS:")
    for c in unclassified_source:
        print("  -", repr(c))

if manifest_not_in_source:
    print("\nMANIFEST COLUMNS NOT FOUND IN SOURCE:")
    for c in manifest_not_in_source:
        print("  -", repr(c))

# Strict frozen-schema policy:
# 1) Nothing in the actual source may be silently ignored.
# 2) Every frozen manifest field must exist in this exact input artifact.
assert not unclassified_source, (
    "Frozen schema violation: source contains unclassified columns. "
    "Review them explicitly before proceeding."
)
assert not manifest_not_in_source, (
    "Frozen schema violation: expected manifest columns are absent from source. "
    "Do not silently continue; inspect upstream schema/naming."
)
assert len(source_set) == len(manifest_set), "Source/manifest column counts differ."

print("\nPASS — every source column is classified exactly once.")


,metric,value
0,source_column_count,110
1,manifest_column_count,110
2,unclassified_source_count,0
3,manifest_not_in_source_count,0



PASS — every source column is classified exactly once.


## 4. Export frozen column manifest before filtering

In [5]:
manifest_rows = []
for order, c in enumerate(source_cols, start=1):
    meta = STATUS[c]
    manifest_rows.append({
        "source_order": order,
        "column": c,
        "status": meta["status"],
        "block": meta["block"],
        "reason": meta["reason"],
        "retained_in_FE_master": meta["status"] != "DROP",
        "default_predictor": meta["status"] == "MODEL_KEEP",
    })

manifest = pd.DataFrame(manifest_rows)

print("Manifest status counts:")
display(
    manifest.groupby(["status"], dropna=False)
    .size()
    .rename("N")
    .reset_index()
    .sort_values("status")
)

display(manifest)

manifest_csv = QA_DIR / "06B_frozen_column_manifest.csv"
manifest_json = QA_DIR / "06B_frozen_column_manifest.json"

manifest.to_csv(manifest_csv, index=False)
manifest_json.write_text(
    json.dumps(manifest.to_dict(orient="records"), indent=2, ensure_ascii=False),
    encoding="utf-8"
)


Manifest status counts:


,status,N
0,CONTROL_QA,3
1,DROP,44
2,KEEP_FE,32
3,MODEL_KEEP,30
4,TARGET,1


,source_order,column,status,block,reason,retained_in_FE_master,default_predictor
0,1,Loan_ID,CONTROL_QA,Block 1,Identity / split control; retained but not a p...,True,False
1,2,Dataset_Vintage,KEEP_FE,Block 1,Time/vintage source retained for downstream fe...,True,False
2,3,FICO_Score,MODEL_KEEP,Block 2,Retained loan-level modeling / challenger feat...,True,True
3,4,First_Payment_Date,KEEP_FE,Block 1,Time/vintage source retained for downstream fe...,True,False
4,5,First_Time_Homebuyer,MODEL_KEEP,Block 2,Retained loan-level modeling / challenger feat...,True,True
5,6,Maturity_Date,KEEP_FE,Block 1,Time/vintage source retained for downstream fe...,True,False
6,7,MSA,KEEP_FE,Block 2,Geography keys required for State/MSA/ZIP-rela...,True,False
7,8,MI_Percentage,MODEL_KEEP,Block 2,Retained loan-level modeling / challenger feat...,True,True
8,9,Number_of_Units,MODEL_KEEP,Block 2,Retained loan-level modeling / challenger feat...,True,True
9,10,Occupancy_Status,MODEL_KEEP,Block 2,Retained loan-level modeling / challenger feat...,True,True


29182

## 5. Row filter: Train + Validation only

In [6]:
split_before = (
    df["Split"].astype("string")
    .value_counts(dropna=False)
    .rename_axis("Split")
    .reset_index(name="N_before")
)

keep_split = df["Split"].astype("string").isin(["Train", "Validation"])
df_tv = df.loc[keep_split].copy()

split_after = (
    df_tv["Split"].astype("string")
    .value_counts(dropna=False)
    .rename_axis("Split")
    .reset_index(name="N_after")
)

split_audit = split_before.merge(split_after, on="Split", how="left")
split_audit["N_after"] = split_audit["N_after"].fillna(0).astype("int64")
split_audit["N_removed"] = split_audit["N_before"] - split_audit["N_after"]

display(split_audit)

assert set(df_tv["Split"].astype("string").unique()) == {"Train", "Validation"}, \
    f"Unexpected retained splits: {df_tv['Split'].unique().tolist()}"
assert "OOT" not in set(df_tv["Split"].astype("string").unique())
assert len(df_tv) == int(
    df["Split"].astype("string").isin(["Train", "Validation"]).sum()
)

print(f"Rows before : {len(df):,}")
print(f"Rows after  : {len(df_tv):,}")
print(f"Rows removed: {len(df)-len(df_tv):,}")


,Split,N_before,N_after,N_removed
0,Other Period,404240,0,404240
1,Train,160003,160003,0
2,OOT Test,99719,0,99719
3,Validation,49038,49038,0


Rows before : 713,000
Rows after  : 209,041
Rows removed: 503,959


## 6. Apply frozen column filter

In [7]:
DROP_COLUMNS = [
    c for c in source_cols if STATUS[c]["status"] == "DROP"
]
KEEP_COLUMNS = [
    c for c in source_cols if STATUS[c]["status"] != "DROP"
]

df_reduced = df_tv.loc[:, KEEP_COLUMNS].copy()

print(f"Columns before : {df.shape[1]:,}")
print(f"Columns kept   : {df_reduced.shape[1]:,}")
print(f"Columns dropped: {len(DROP_COLUMNS):,}")

print("\nDropped columns:")
for c in DROP_COLUMNS:
    print("  -", c)

# Structural assertions
assert list(df_reduced.columns) == KEEP_COLUMNS, \
    "Retained column order differs from frozen source-order manifest."
assert not (set(DROP_COLUMNS) & set(df_reduced.columns)), \
    "At least one DROP column remains."
assert set(df_reduced.columns) == set(KEEP_COLUMNS), \
    "Reduced schema differs from frozen KEEP schema."


Columns before : 110
Columns kept   : 66
Columns dropped: 44

Dropped columns:
  - Prepayment_Penalty_Indicator
  - Amortization_Type
  - Special_Eligibility_Program
  - Property_Valuation_Method
  - Interest_Only_Indicator
  - Freddie_State
  - Freddie_MSA
  - Freddie_ZIP3
  - FHFA_ZIP3_HPI
  - FHFA_ZIP3_HPI_Growth_12M
  - BLS_Unemployment_Rate
  - BLS_Employment
  - BLS_Labor_Force
  - BLS_Employment_Growth_12M
  - BLS_Unemployment_Rate_Change_12M
  - Labor_Geography_Source
  - LS_Freddie_State
  - LS_Freddie_ZIP3
  - LS_Freddie_MSA
  - LS_FHFA_ZIP3_HPI
  - LS_FHFA_ZIP3_HPI_Growth_12M
  - LS_BLS_Unemployment_Rate
  - LS_BLS_Employment
  - LS_BLS_Labor_Force
  - LS_BLS_Employment_Growth_12M
  - LS_BLS_Unemployment_Rate_Change_12M
  - LS_Labor_Geography_Source
  - _State_FIPS
  - Strict_PIT_BLS_State_Month_Date
  - BLS_State_Assumed_Available_Date
  - _State_BLS_Unemployment_Rate
  - _State_BLS_Civilian_Labor_Force
  - _State_BLS_Employment
  - _State_BLS_Unemployment
  - _State_BLS_La

## 7. Downstream-required feature-family assertions

In [8]:
REGULAR_FINAL_MACRO = [
    "Housing_HPI",
    "Housing_HPI_Growth_12M",
    "BLS_Unemployment_Rate_Final",
    "BLS_Employment_Growth_12M_Final",
    "BLS_Unemployment_Rate_Change_12M_Final",
]

STRICT_FINAL_MACRO = [
    "LS_Housing_HPI",
    "LS_Housing_HPI_Growth_12M",
    "LS_BLS_Unemployment_Rate_Final",
    "LS_BLS_Employment_Growth_12M_Final",
    "LS_BLS_Unemployment_Rate_Change_12M_Final",
]

STATE_MSA_FE_INPUTS = [
    "Property_State", "MSA",
    "PIT_FHFA_Quarter_Date", "Strict_PIT_FHFA_Quarter_Date",
    "FHFA_State_HPI", "FHFA_State_HPI_Growth_12M",
    "FHFA_MSA_HPI", "FHFA_MSA_HPI_Growth_12M",
    "LS_FHFA_State_HPI", "LS_FHFA_State_HPI_Growth_12M",
    "LS_FHFA_MSA_HPI", "LS_FHFA_MSA_HPI_Growth_12M",
]

LOAN_FE_INPUTS = [
    "FICO_Score",
    "Original_DTI",
    "DTI_Missing",
    "Original_LTV",
    "Original_CLTV",
    "Original_UPB",
    "Original_Interest_Rate",
    "Original_Loan_Term",
    "Number_of_Borrowers",
    "HARP_Indicator",
]

RESIDUAL_FLAGS = [
    "LS_Housing_ResidualMissing",
    "LS_BLS_ResidualMissing",
    "Housing_Regular_ResidualMissing",
    "BLS_Regular_ResidualMissing",
]

CONTROL_REQUIRED = [
    "Loan_ID", "Split", "Target_24M", "Performance_Window_End"
]

required_groups = {
    "Regular final macro": REGULAR_FINAL_MACRO,
    "Strict final macro": STRICT_FINAL_MACRO,
    "State/MSA FE inputs": STATE_MSA_FE_INPUTS,
    "Loan FE inputs": LOAN_FE_INPUTS,
    "Residual missing flags": RESIDUAL_FLAGS,
    "Control/QA": CONTROL_REQUIRED,
}

required_audit = []
for group, cols in required_groups.items():
    for c in cols:
        required_audit.append({
            "group": group,
            "column": c,
            "present": c in df_reduced.columns,
        })

required_audit = pd.DataFrame(required_audit)
display(required_audit)

assert required_audit["present"].all(), \
    "At least one required downstream field was lost during reduction."

print("PASS — all frozen downstream feature families remain available.")


,group,column,present
0,Regular final macro,Housing_HPI,True
1,Regular final macro,Housing_HPI_Growth_12M,True
2,Regular final macro,BLS_Unemployment_Rate_Final,True
3,Regular final macro,BLS_Employment_Growth_12M_Final,True
4,Regular final macro,BLS_Unemployment_Rate_Change_12M_Final,True
5,Strict final macro,LS_Housing_HPI,True
6,Strict final macro,LS_Housing_HPI_Growth_12M,True
7,Strict final macro,LS_BLS_Unemployment_Rate_Final,True
8,Strict final macro,LS_BLS_Employment_Growth_12M_Final,True
9,Strict final macro,LS_BLS_Unemployment_Rate_Change_12M_Final,True


PASS — all frozen downstream feature families remain available.


## 8. Post-filter invariants + missingness audit

In [9]:
# Identity / target / split invariants
assert df_reduced["Loan_ID"].notna().all()
assert df_reduced["Loan_ID"].is_unique

target_numeric = pd.to_numeric(df_reduced["Target_24M"], errors="coerce")
assert target_numeric.notna().all()
assert set(target_numeric.unique()).issubset({0, 1})

retained_splits = set(df_reduced["Split"].astype("string").unique())
assert retained_splits == {"Train", "Validation"}
assert "OOT" not in retained_splits

# No new missing-value treatment is performed here.
# This audit records the retained schema exactly as it exists.
missing_audit = pd.DataFrame({
    "column": df_reduced.columns,
    "dtype": [str(df_reduced[c].dtype) for c in df_reduced.columns],
    "missing_n": [int(df_reduced[c].isna().sum()) for c in df_reduced.columns],
})
missing_audit["missing_pct"] = (
    missing_audit["missing_n"] / len(df_reduced) * 100
)
missing_audit = missing_audit.sort_values(
    ["missing_pct", "column"], ascending=[False, True]
).reset_index(drop=True)

print("Retained-column missingness audit:")
display(missing_audit)

missing_audit.to_csv(
    QA_DIR / "06B_retained_columns_missingness.csv", index=False
)
split_audit.to_csv(
    QA_DIR / "06B_split_filter_audit.csv", index=False
)
required_audit.to_csv(
    QA_DIR / "06B_required_feature_family_audit.csv", index=False
)


Retained-column missingness audit:


,column,dtype,missing_n,missing_pct
0,FHFA_MSA_HPI,float64,40394,19.323482
1,FHFA_MSA_HPI_Growth_12M,float64,40394,19.323482
2,LS_FHFA_MSA_HPI,float64,40394,19.323482
3,LS_FHFA_MSA_HPI_Growth_12M,float64,40394,19.323482
4,FHFA_State_HPI,float64,99,0.047359
5,FHFA_State_HPI_Growth_12M,float64,99,0.047359
6,Housing_Fallback_Level,Int64,99,0.047359
7,Housing_Geography_Source,string,99,0.047359
8,LS_FHFA_State_HPI,float64,99,0.047359
9,LS_FHFA_State_HPI_Growth_12M,float64,99,0.047359


## 9. Train vs Validation target audit (descriptive only)

In [10]:
target_by_split = (
    df_reduced
    .assign(Target_24M=pd.to_numeric(df_reduced["Target_24M"], errors="raise"))
    .groupby("Split", observed=False)["Target_24M"]
    .agg(N="size", Events="sum", Target_Rate="mean")
    .reset_index()
)
target_by_split["Events"] = target_by_split["Events"].astype("int64")

display(target_by_split)
target_by_split.to_csv(
    QA_DIR / "06B_target_by_split.csv", index=False
)


,Split,N,Events,Target_Rate
0,Train,160003,919,0.005744
1,Validation,49038,380,0.007749


## 10. Save FE-master parquet + QA manifest

In [11]:
# Final pre-save assertions
assert len(df_reduced) == len(df_tv)
assert df_reduced.shape[1] == len(KEEP_COLUMNS)
assert not any(c in df_reduced.columns for c in DROP_COLUMNS)
assert df_reduced["Loan_ID"].is_unique

OUTPUT.parent.mkdir(parents=True, exist_ok=True)
df_reduced.to_parquet(OUTPUT, index=False)

summary = {
    "stage": "06B_train_validation_row_filter_and_FE_master_column_freeze",
    "source_artifact": str(INPUT.relative_to(ROOT)),
    "output_artifact": str(OUTPUT.relative_to(ROOT)),
    "source_shape": [int(df.shape[0]), int(df.shape[1])],
    "output_shape": [int(df_reduced.shape[0]), int(df_reduced.shape[1])],
    "row_rule": "Split in {'Train', 'Validation'}",
    "excluded_split": "OOT",
    "feature_engineering_performed": False,
    "imputation_performed": False,
    "scaling_performed": False,
    "clustering_performed": False,
    "regular_PIT_preserved": True,
    "strict_PIT_preserved": True,
    "source_column_count": int(len(source_cols)),
    "kept_column_count": int(len(KEEP_COLUMNS)),
    "dropped_column_count": int(len(DROP_COLUMNS)),
    "kept_columns_in_source_order": KEEP_COLUMNS,
    "dropped_columns_in_source_order": DROP_COLUMNS,
    "status_counts": {
        k: int(v)
        for k, v in manifest["status"].value_counts().to_dict().items()
    },
}

summary_path = QA_DIR / "06B_reduction_summary.json"
summary_path.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("Saved:")
print(" - FE master :", OUTPUT)
print(" - Manifest  :", manifest_csv)
print(" - Summary   :", summary_path)
print("\nFinal shape:", df_reduced.shape)


Saved:
 - FE master : C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6\data\output\final_modeling_dataset\SFLLD_feature_engineering_master_train_validation.parquet
 - Manifest  : C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6\data\output\quality_assurance\06B_train_validation_FE_master_freeze\06B_frozen_column_manifest.csv
 - Summary   : C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6\data\output\quality_assurance\06B_train_validation_FE_master_freeze\06B_reduction_summary.json

Final shape: (209041, 66)


## 11. Reload verification — verify the actual saved artifact

In [12]:
check = pd.read_parquet(OUTPUT)

assert check.shape == df_reduced.shape, \
    f"Reloaded shape differs: saved={check.shape}, expected={df_reduced.shape}"
assert list(check.columns) == list(df_reduced.columns), \
    "Reloaded column order/schema differs."
assert check["Loan_ID"].notna().all()
assert check["Loan_ID"].is_unique
assert set(check["Split"].astype("string").unique()) == {"Train", "Validation"}
assert not any(c in check.columns for c in DROP_COLUMNS)

# Exact Loan_ID/order preservation through save/reload
assert check["Loan_ID"].astype("string").equals(
    df_reduced["Loan_ID"].astype("string").reset_index(drop=True)
), "Loan_ID/order changed after save/reload."

print("=" * 100)
print("06B COMPLETE — TRAIN/VALIDATION FE-MASTER DATASET FROZEN")
print("=" * 100)
print(f"Source rows        : {df.shape[0]:,}")
print(f"Output rows        : {check.shape[0]:,}")
print(f"Source columns     : {df.shape[1]:,}")
print(f"Output columns     : {check.shape[1]:,}")
print(f"Dropped columns    : {len(DROP_COLUMNS):,}")
print("Retained splits    :", sorted(check["Split"].astype("string").unique().tolist()))
print("Regular PIT kept   : YES")
print("Strict PIT kept    : YES")
print("New FE created     : NO")
print("Source overwritten : NO")
print("\nOutput:")
print(OUTPUT)
print("\nQA:")
print(QA_DIR)


06B COMPLETE — TRAIN/VALIDATION FE-MASTER DATASET FROZEN
Source rows        : 713,000
Output rows        : 209,041
Source columns     : 110
Output columns     : 66
Dropped columns    : 44
Retained splits    : ['Train', 'Validation']
Regular PIT kept   : YES
Strict PIT kept    : YES
New FE created     : NO
Source overwritten : NO

Output:
C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6\data\output\final_modeling_dataset\SFLLD_feature_engineering_master_train_validation.parquet

QA:
C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6\data\output\quality_assurance\06B_train_validation_FE_master_freeze


In [13]:
# ======================================================================================
# 06B — POST-SAVE MISSING-VALUE AUDIT
# ======================================================================================

import pandas as pd
import numpy as np
from IPython.display import display

print("=" * 100)
print("06B POST-SAVE MISSING-VALUE AUDIT")
print("=" * 100)

# --------------------------------------------------------------------------------------
# 1. Load the SAVED FE-master dataset
# --------------------------------------------------------------------------------------

final_path = (
    ROOT
    / "data"
    / "output"
    / "final_modeling_dataset"
    / "SFLLD_feature_engineering_master_train_validation.parquet"
)

assert final_path.exists(), f"Final FE-master parquet not found:\n{final_path}"

df_check = pd.read_parquet(final_path)

print(f"\nLoaded : {final_path}")
print(f"Shape  : {df_check.shape[0]:,} rows × {df_check.shape[1]:,} columns")


# --------------------------------------------------------------------------------------
# 2. Overall missing-value audit
# --------------------------------------------------------------------------------------

missing_count = df_check.isna().sum()
missing_pct = missing_count / len(df_check) * 100

missing_audit = pd.DataFrame({
    "Column": df_check.columns,
    "Missing_Count": missing_count.values,
    "Missing_Pct": missing_pct.values,
    "Dtype": df_check.dtypes.astype(str).values,
})

missing_audit = (
    missing_audit
    .sort_values(
        ["Missing_Count", "Column"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

columns_with_missing = missing_audit[
    missing_audit["Missing_Count"] > 0
].copy()

total_missing_cells = int(missing_count.sum())
n_columns_with_missing = int((missing_count > 0).sum())

print("\n" + "-" * 100)
print("OVERALL MISSING-VALUE SUMMARY")
print("-" * 100)

print(f"Rows                        : {len(df_check):,}")
print(f"Columns                     : {df_check.shape[1]:,}")
print(f"Columns with missing values : {n_columns_with_missing:,}")
print(f"Total missing cells         : {total_missing_cells:,}")

if n_columns_with_missing == 0:
    print("\nPASS — No pandas-recognized missing values remain.")
else:
    print("\nWARNING — Missing values remain:")
    display(columns_with_missing)


# --------------------------------------------------------------------------------------
# 3. Missing values by Split
# --------------------------------------------------------------------------------------

assert "Split" in df_check.columns, "'Split' column is missing."

split_records = []

for split_name, g in df_check.groupby("Split", observed=True):

    split_missing = g.isna().sum()

    for col in df_check.columns:

        n_missing = int(split_missing[col])

        if n_missing > 0:
            split_records.append({
                "Split": split_name,
                "Column": col,
                "N_Rows": len(g),
                "Missing_Count": n_missing,
                "Missing_Pct": n_missing / len(g) * 100,
                "Dtype": str(g[col].dtype),
            })

split_missing_audit = pd.DataFrame(split_records)

print("\n" + "-" * 100)
print("SPLIT-SPECIFIC MISSING VALUES")
print("-" * 100)

if split_missing_audit.empty:
    print("PASS — No NaN values in either Train or Validation.")
else:
    split_missing_audit = (
        split_missing_audit
        .sort_values(
            ["Split", "Missing_Count"],
            ascending=[True, False]
        )
        .reset_index(drop=True)
    )

    display(split_missing_audit)


# --------------------------------------------------------------------------------------
# 4. Explicit missing-like CATEGORY audit
#
# Important:
# "Missing", "NA", etc. may be intentional category values.
# They are NOT treated as NaN here.
# --------------------------------------------------------------------------------------

MISSING_LIKE_LABELS = {
    "",
    "missing",
    "nan",
    "none",
    "null",
    "na",
    "n/a",
    "<na>",
}

categorical_records = []

for col in df_check.columns:

    s = df_check[col]

    if (
        pd.api.types.is_object_dtype(s.dtype)
        or isinstance(s.dtype, pd.CategoricalDtype)
        or pd.api.types.is_string_dtype(s.dtype)
    ):

        normalized = (
            s.astype("string")
            .str.strip()
            .str.lower()
        )

        mask = normalized.isin(MISSING_LIKE_LABELS)

        if mask.any():

            categorical_records.append({
                "Column": col,
                "Missing_Like_Count": int(mask.sum()),
                "Missing_Like_Pct": float(mask.mean() * 100),
                "Observed_Labels": sorted(
                    s.loc[mask]
                    .astype("string")
                    .dropna()
                    .unique()
                    .tolist()
                )
            })

categorical_missing_like = pd.DataFrame(categorical_records)

print("\n" + "-" * 100)
print("EXPLICIT MISSING-LIKE CATEGORY AUDIT")
print("-" * 100)

if categorical_missing_like.empty:
    print("No explicit missing-like category labels detected.")
else:
    print(
        "NOTE: These are category/string values, NOT pandas NaN.\n"
        "They may be intentional and should not automatically be imputed."
    )
    display(categorical_missing_like)


# --------------------------------------------------------------------------------------
# 5. Save QA artifacts
# --------------------------------------------------------------------------------------

missing_audit.to_csv(
    QA_DIR / "06B_final_FE_master_missing_value_audit_all_columns.csv",
    index=False
)

columns_with_missing.to_csv(
    QA_DIR / "06B_final_FE_master_columns_with_missing.csv",
    index=False
)

if not split_missing_audit.empty:
    split_missing_audit.to_csv(
        QA_DIR / "06B_final_FE_master_missing_by_split.csv",
        index=False
    )

if not categorical_missing_like.empty:
    categorical_missing_like.to_csv(
        QA_DIR / "06B_final_FE_master_explicit_missing_like_categories.csv",
        index=False
    )


# --------------------------------------------------------------------------------------
# 6. Final verdict
# --------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("06B MISSING-VALUE AUDIT COMPLETE")
print("=" * 100)

print(f"Dataset                : {final_path.name}")
print(f"Rows                   : {len(df_check):,}")
print(f"Columns                : {df_check.shape[1]:,}")
print(f"NaN-containing columns : {n_columns_with_missing:,}")
print(f"Total NaN cells        : {total_missing_cells:,}")

if total_missing_cells == 0:
    print("\nFINAL NaN VERDICT: PASS")
    print("No pandas-recognized NaN values remain in the saved FE-master dataset.")
else:
    print("\nFINAL NaN VERDICT: REVIEW REQUIRED")
    print("The saved FE-master still contains pandas-recognized missing values.")

print(
    "\nExplicit missing-like category labels are audited separately "
    "and are not automatically considered errors."
)

06B POST-SAVE MISSING-VALUE AUDIT

Loaded : C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6\data\output\final_modeling_dataset\SFLLD_feature_engineering_master_train_validation.parquet
Shape  : 209,041 rows × 66 columns

----------------------------------------------------------------------------------------------------
OVERALL MISSING-VALUE SUMMARY
----------------------------------------------------------------------------------------------------
Rows                        : 209,041
Columns                     : 66
Columns with missing values : 12
Total missing cells         : 162,368

WARNING — Missing values remain:


,Column,Missing_Count,Missing_Pct,Dtype
0,FHFA_MSA_HPI,40394,19.323482,float64
1,FHFA_MSA_HPI_Growth_12M,40394,19.323482,float64
2,LS_FHFA_MSA_HPI,40394,19.323482,float64
3,LS_FHFA_MSA_HPI_Growth_12M,40394,19.323482,float64
4,FHFA_State_HPI,99,0.047359,float64
5,FHFA_State_HPI_Growth_12M,99,0.047359,float64
6,Housing_Fallback_Level,99,0.047359,Int64
7,Housing_Geography_Source,99,0.047359,string
8,LS_FHFA_State_HPI,99,0.047359,float64
9,LS_FHFA_State_HPI_Growth_12M,99,0.047359,float64



----------------------------------------------------------------------------------------------------
SPLIT-SPECIFIC MISSING VALUES
----------------------------------------------------------------------------------------------------


,Split,Column,N_Rows,Missing_Count,Missing_Pct,Dtype
0,Train,FHFA_MSA_HPI,160003,31806,19.878377,float64
1,Train,FHFA_MSA_HPI_Growth_12M,160003,31806,19.878377,float64
2,Train,LS_FHFA_MSA_HPI,160003,31806,19.878377,float64
3,Train,LS_FHFA_MSA_HPI_Growth_12M,160003,31806,19.878377,float64
4,Train,FHFA_State_HPI,160003,86,0.053749,float64
5,Train,FHFA_State_HPI_Growth_12M,160003,86,0.053749,float64
6,Train,Housing_Geography_Source,160003,86,0.053749,string
7,Train,Housing_Fallback_Level,160003,86,0.053749,Int64
8,Train,LS_FHFA_State_HPI,160003,86,0.053749,float64
9,Train,LS_FHFA_State_HPI_Growth_12M,160003,86,0.053749,float64



----------------------------------------------------------------------------------------------------
EXPLICIT MISSING-LIKE CATEGORY AUDIT
----------------------------------------------------------------------------------------------------
NOTE: These are category/string values, NOT pandas NaN.
They may be intentional and should not automatically be imputed.


,Column,Missing_Like_Count,Missing_Like_Pct,Observed_Labels
0,First_Time_Homebuyer,13,0.006219,[Missing]
1,MSA,24064,11.511617,[Missing]
2,LS_BLS_Geography_Source_Final,87,0.041619,[MISSING]
3,LS_BLS_Fallback_Level,87,0.041619,[MISSING]



06B MISSING-VALUE AUDIT COMPLETE
Dataset                : SFLLD_feature_engineering_master_train_validation.parquet
Rows                   : 209,041
Columns                : 66
NaN-containing columns : 12
Total NaN cells        : 162,368

FINAL NaN VERDICT: REVIEW REQUIRED
The saved FE-master still contains pandas-recognized missing values.

Explicit missing-like category labels are audited separately and are not automatically considered errors.


In [14]:
# ======================================================================================
# 06B — COMPLETE MISSINGNESS AUDIT FOR ALL RETAINED VARIABLES
# ======================================================================================

import pandas as pd
import numpy as np
from IPython.display import display

print("=" * 110)
print("06B — COMPLETE MISSINGNESS AUDIT: ALL RETAINED VARIABLES")
print("=" * 110)

# --------------------------------------------------------------------------------------
# 1. Load the SAVED Train/Validation FE-master
# --------------------------------------------------------------------------------------

final_path = (
    ROOT
    / "data"
    / "output"
    / "final_modeling_dataset"
    / "SFLLD_feature_engineering_master_train_validation.parquet"
)

assert final_path.exists(), f"File not found:\n{final_path}"

df_check = pd.read_parquet(final_path)

assert "Split" in df_check.columns

print(f"\nDataset : {final_path.name}")
print(f"Shape   : {df_check.shape[0]:,} rows × {df_check.shape[1]:,} columns")

print("\nSplit distribution:")
display(
    df_check["Split"]
    .value_counts(dropna=False)
    .rename_axis("Split")
    .reset_index(name="N")
)


# ======================================================================================
# 2. Overall missingness — ALL columns
# ======================================================================================

N_TOTAL = len(df_check)

overall_missing_n = df_check.isna().sum()

overall_table = pd.DataFrame({
    "Column": df_check.columns,
    "Dtype": [str(df_check[c].dtype) for c in df_check.columns],
    "N_Total": N_TOTAL,
    "Missing_N": [int(overall_missing_n[c]) for c in df_check.columns],
})

overall_table["Missing_Pct"] = (
    overall_table["Missing_N"] / overall_table["N_Total"] * 100
)

overall_table["Observed_N"] = (
    overall_table["N_Total"] - overall_table["Missing_N"]
)

overall_table["Observed_Pct"] = (
    overall_table["Observed_N"] / overall_table["N_Total"] * 100
)


# ======================================================================================
# 3. Train-specific missingness
# ======================================================================================

train = df_check.loc[df_check["Split"].astype("string") == "Train"]

N_TRAIN = len(train)

train_missing_n = train.isna().sum()

overall_table["Train_N"] = N_TRAIN
overall_table["Train_Missing_N"] = [
    int(train_missing_n[c]) for c in df_check.columns
]

overall_table["Train_Missing_Pct"] = (
    overall_table["Train_Missing_N"]
    / overall_table["Train_N"]
    * 100
)


# ======================================================================================
# 4. Validation-specific missingness
# ======================================================================================

validation = df_check.loc[
    df_check["Split"].astype("string") == "Validation"
]

N_VALIDATION = len(validation)

validation_missing_n = validation.isna().sum()

overall_table["Validation_N"] = N_VALIDATION
overall_table["Validation_Missing_N"] = [
    int(validation_missing_n[c]) for c in df_check.columns
]

overall_table["Validation_Missing_Pct"] = (
    overall_table["Validation_Missing_N"]
    / overall_table["Validation_N"]
    * 100
)


# ======================================================================================
# 5. Missingness difference: Validation − Train
# ======================================================================================

overall_table["Val_minus_Train_Missing_Pct_pp"] = (
    overall_table["Validation_Missing_Pct"]
    - overall_table["Train_Missing_Pct"]
)


# ======================================================================================
# 6. Sort by overall missingness
# ======================================================================================

overall_table = (
    overall_table
    .sort_values(
        ["Missing_Pct", "Column"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

overall_table.insert(
    0,
    "Rank",
    np.arange(1, len(overall_table) + 1)
)


# ======================================================================================
# 7. Display ALL variables
# ======================================================================================

print("\n" + "-" * 110)
print("ALL RETAINED VARIABLES — MISSINGNESS TABLE")
print("-" * 110)

display(
    overall_table[
        [
            "Rank",
            "Column",
            "Dtype",

            "N_Total",
            "Missing_N",
            "Missing_Pct",

            "Train_N",
            "Train_Missing_N",
            "Train_Missing_Pct",

            "Validation_N",
            "Validation_Missing_N",
            "Validation_Missing_Pct",

            "Val_minus_Train_Missing_Pct_pp",
        ]
    ]
)


# ======================================================================================
# 8. Variables WITH actual NaN
# ======================================================================================

with_nan = overall_table.loc[
    overall_table["Missing_N"] > 0
].copy()

print("\n" + "-" * 110)
print("VARIABLES WITH ACTUAL NaN")
print("-" * 110)

if len(with_nan) == 0:
    print("None.")
else:
    display(
        with_nan[
            [
                "Column",
                "Missing_N",
                "Missing_Pct",
                "Train_Missing_N",
                "Train_Missing_Pct",
                "Validation_Missing_N",
                "Validation_Missing_Pct",
            ]
        ]
    )


# ======================================================================================
# 9. Variables with ZERO actual NaN
# ======================================================================================

zero_nan = overall_table.loc[
    overall_table["Missing_N"] == 0
].copy()

print("\n" + "-" * 110)
print("VARIABLES WITH ZERO ACTUAL NaN")
print("-" * 110)

display(
    zero_nan[
        [
            "Column",
            "Dtype",
            "Missing_N",
            "Missing_Pct",
        ]
    ]
)


# ======================================================================================
# 10. Explicit missing-like categorical values
#
# IMPORTANT:
# These are values such as "Missing" or "MISSING".
# They are NOT pandas NaN, so inspect them separately.
# ======================================================================================

MISSING_LIKE_LABELS = {
    "",
    "missing",
    "nan",
    "none",
    "null",
    "na",
    "n/a",
    "<na>",
}

explicit_records = []

for col in df_check.columns:

    s = df_check[col]

    if (
        pd.api.types.is_object_dtype(s.dtype)
        or pd.api.types.is_string_dtype(s.dtype)
        or isinstance(s.dtype, pd.CategoricalDtype)
    ):

        normalized = (
            s.astype("string")
            .str.strip()
            .str.lower()
        )

        explicit_mask = normalized.isin(MISSING_LIKE_LABELS)

        if explicit_mask.any():

            # Overall
            overall_n = int(explicit_mask.sum())

            # Train
            train_mask = (
                df_check["Split"].astype("string") == "Train"
            )
            train_explicit_n = int(
                (explicit_mask & train_mask).sum()
            )

            # Validation
            val_mask = (
                df_check["Split"].astype("string") == "Validation"
            )
            val_explicit_n = int(
                (explicit_mask & val_mask).sum()
            )

            labels = sorted(
                s.loc[explicit_mask]
                .astype("string")
                .dropna()
                .unique()
                .tolist()
            )

            explicit_records.append({
                "Column": col,
                "Labels": labels,

                "Overall_N": overall_n,
                "Overall_Pct": overall_n / N_TOTAL * 100,

                "Train_N": train_explicit_n,
                "Train_Pct": (
                    train_explicit_n / N_TRAIN * 100
                    if N_TRAIN > 0 else np.nan
                ),

                "Validation_N": val_explicit_n,
                "Validation_Pct": (
                    val_explicit_n / N_VALIDATION * 100
                    if N_VALIDATION > 0 else np.nan
                ),
            })


explicit_missing_table = pd.DataFrame(explicit_records)

print("\n" + "-" * 110)
print("EXPLICIT MISSING-LIKE CATEGORIES")
print("-" * 110)

if explicit_missing_table.empty:
    print("No explicit missing-like category values detected.")
else:
    explicit_missing_table = (
        explicit_missing_table
        .sort_values(
            "Overall_Pct",
            ascending=False
        )
        .reset_index(drop=True)
    )

    display(explicit_missing_table)


# ======================================================================================
# 11. Combined summary
# ======================================================================================

print("\n" + "=" * 110)
print("MISSINGNESS SUMMARY")
print("=" * 110)

print(f"Rows                                   : {N_TOTAL:,}")
print(f"Columns                                : {df_check.shape[1]:,}")

print(f"\nTrain rows                             : {N_TRAIN:,}")
print(f"Validation rows                        : {N_VALIDATION:,}")

print(
    f"\nColumns with actual NaN                : "
    f"{int((overall_table['Missing_N'] > 0).sum()):,}"
)

print(
    f"Columns with zero actual NaN           : "
    f"{int((overall_table['Missing_N'] == 0).sum()):,}"
)

print(
    f"Total actual NaN cells                 : "
    f"{int(df_check.isna().sum().sum()):,}"
)

print(
    f"Overall actual NaN cell rate           : "
    f"{df_check.isna().sum().sum() / df_check.size * 100:.6f}%"
)

if not explicit_missing_table.empty:
    print(
        f"Columns with explicit missing category : "
        f"{len(explicit_missing_table):,}"
    )


# ======================================================================================
# 12. Save QA tables
# ======================================================================================

all_path = (
    QA_DIR
    / "06B_ALL_66_variables_missingness.csv"
)

nan_path = (
    QA_DIR
    / "06B_variables_with_actual_NaN.csv"
)

zero_path = (
    QA_DIR
    / "06B_variables_with_zero_NaN.csv"
)

explicit_path = (
    QA_DIR
    / "06B_explicit_missing_category_audit.csv"
)

overall_table.to_csv(
    all_path,
    index=False
)

with_nan.to_csv(
    nan_path,
    index=False
)

zero_nan.to_csv(
    zero_path,
    index=False
)

explicit_missing_table.to_csv(
    explicit_path,
    index=False
)


print("\nSaved QA files:")
print(" -", all_path)
print(" -", nan_path)
print(" -", zero_path)
print(" -", explicit_path)

print("\n" + "=" * 110)
print("06B COMPLETE ALL-VARIABLE MISSINGNESS AUDIT FINISHED")
print("=" * 110)

06B — COMPLETE MISSINGNESS AUDIT: ALL RETAINED VARIABLES

Dataset : SFLLD_feature_engineering_master_train_validation.parquet
Shape   : 209,041 rows × 66 columns

Split distribution:


,Split,N
0,Train,160003
1,Validation,49038



--------------------------------------------------------------------------------------------------------------
ALL RETAINED VARIABLES — MISSINGNESS TABLE
--------------------------------------------------------------------------------------------------------------


,Rank,Column,Dtype,N_Total,Missing_N,Missing_Pct,Train_N,Train_Missing_N,Train_Missing_Pct,Validation_N,Validation_Missing_N,Validation_Missing_Pct,Val_minus_Train_Missing_Pct_pp
0,1,FHFA_MSA_HPI,float64,209041,40394,19.323482,160003,31806,19.878377,49038,8588,17.512949,-2.365428
1,2,FHFA_MSA_HPI_Growth_12M,float64,209041,40394,19.323482,160003,31806,19.878377,49038,8588,17.512949,-2.365428
2,3,LS_FHFA_MSA_HPI,float64,209041,40394,19.323482,160003,31806,19.878377,49038,8588,17.512949,-2.365428
3,4,LS_FHFA_MSA_HPI_Growth_12M,float64,209041,40394,19.323482,160003,31806,19.878377,49038,8588,17.512949,-2.365428
4,5,FHFA_State_HPI,float64,209041,99,0.047359,160003,86,0.053749,49038,13,0.026510,-0.027239
5,6,FHFA_State_HPI_Growth_12M,float64,209041,99,0.047359,160003,86,0.053749,49038,13,0.026510,-0.027239
6,7,Housing_Fallback_Level,Int64,209041,99,0.047359,160003,86,0.053749,49038,13,0.026510,-0.027239
7,8,Housing_Geography_Source,string,209041,99,0.047359,160003,86,0.053749,49038,13,0.026510,-0.027239
8,9,LS_FHFA_State_HPI,float64,209041,99,0.047359,160003,86,0.053749,49038,13,0.026510,-0.027239
9,10,LS_FHFA_State_HPI_Growth_12M,float64,209041,99,0.047359,160003,86,0.053749,49038,13,0.026510,-0.027239



--------------------------------------------------------------------------------------------------------------
VARIABLES WITH ACTUAL NaN
--------------------------------------------------------------------------------------------------------------


...


--------------------------------------------------------------------------------------------------------------
VARIABLES WITH ZERO ACTUAL NaN
--------------------------------------------------------------------------------------------------------------


,Column,Dtype,Missing_N,Missing_Pct
12,BLS_Employment_Growth_12M_Final,float64,0,0.0
13,BLS_Fallback_Level,Int8,0,0.0
14,BLS_Geography_Source_Final,string,0,0.0
15,BLS_Regular_ResidualMissing,int8,0,0.0
16,BLS_State_Fallback_Applied,Int8,0,0.0
17,BLS_Unemployment_Rate_Change_12M_Final,float64,0,0.0
18,BLS_Unemployment_Rate_Final,float64,0,0.0
19,COVID_Window_Overlap,bool,0,0.0
20,Channel,string,0,0.0
21,DTI_Missing,int8,0,0.0



--------------------------------------------------------------------------------------------------------------
EXPLICIT MISSING-LIKE CATEGORIES
--------------------------------------------------------------------------------------------------------------


,Column,Labels,Overall_N,Overall_Pct,Train_N,Train_Pct,Validation_N,Validation_Pct
0,MSA,[Missing],24064,11.511617,18851,11.781654,5213,10.630531
1,LS_BLS_Geography_Source_Final,[MISSING],87,0.041619,75,0.046874,12,0.024471
2,LS_BLS_Fallback_Level,[MISSING],87,0.041619,75,0.046874,12,0.024471
3,First_Time_Homebuyer,[Missing],13,0.006219,13,0.008125,0,0.000000



MISSINGNESS SUMMARY
Rows                                   : 209,041
Columns                                : 66

Train rows                             : 160,003
Validation rows                        : 49,038

Columns with actual NaN                : 12
Columns with zero actual NaN           : 54
Total actual NaN cells                 : 162,368
Overall actual NaN cell rate           : 1.176861%
Columns with explicit missing category : 4

Saved QA files:
 - C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6\data\output\quality_assurance\06B_train_validation_FE_master_freeze\06B_ALL_66_variables_missingness.csv
 - C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6\data\output\quality_assurance\06B_train_validation_FE_master_freeze\06B_variables_with_actual_NaN.csv
 - C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6\data\output\quality_assurance\06B_train_validation_FE_ma